In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# patten-server GPU: use PyTorch cu124 (matches driver CUDA 12.x)
#   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import (
    train_tabddpm,
    train_forestdiffusion,
    resolve_experiment_device,
    print_experiment_runtime,
)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load CDC Diabetes Health Indicators dataset
# ----------------------------------------------------
cdc_diabetes_health_indicators = fetch_ucirepo(id=891)
X = cdc_diabetes_health_indicators.data.features
y = cdc_diabetes_health_indicators.data.targets

print(cdc_diabetes_health_indicators.metadata)
print(cdc_diabetes_health_indicators.variables)

data = pd.concat([X, y], axis=1)
target_col = "Diabetes_binary"

if "ID" in data.columns:
    data = data.drop(columns=["ID"])

# Binary target (0/1)
data[target_col] = pd.to_numeric(data[target_col], errors="coerce")
data = data.dropna(subset=[target_col])
data[target_col] = data[target_col].round().astype(int)

# Ensure numeric feature types
for col in data.columns:
    if col == target_col:
        continue
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

DEVICE = "auto"
EXPERIMENT_DEVICE = resolve_experiment_device(DEVICE)

FAST_MODE = True
RUN_QUALITY_EVAL = True
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
GENERATORS_TO_EVAL = ["ForestDiffusion", "TabDDPM"]

# Benchmark subsample + leak-safe split (generators fit train only)
_bench_n = min(N_SAMPLES, len(data))
data = data.sample(n=_bench_n, random_state=SEED).reset_index(drop=True)

_strat = data[target_col] if data[target_col].nunique() <= 30 else None
train_real, test_real = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=_strat,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)

# Alias used by downstream evaluation cells
cdc_diabetes_data = data

SYNTHETIC_N = N_SAMPLES
DIFFUSION_SEED = SEED
_categorical_columns = [target_col]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
synthetic_outputs = {}
quality_results = []

print(f"Subsample: {data.shape} | train_real: {train_real.shape} | test_real: {test_real.shape}")
print(
    "Class balance (Diabetes_binary=1): "
    f"train {train_real[target_col].mean():.3f} | test {test_real[target_col].mean():.3f}"
)
print_experiment_runtime(EXPERIMENT_DEVICE)

{'uci_id': 891, 'name': 'CDC Diabetes Health Indicators', 'repository_url': 'https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators', 'data_url': 'https://archive.ics.uci.edu/static/public/891/data.csv', 'abstract': 'The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy. ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 253680, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Sex', 'Age', 'Education Level', 'Income'], 'target_col': ['Diabetes_binary'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_

In [3]:
# ---------------------------------------------------
# Diffusion models (TabDDPM, ForestDiffusion)
# ---------------------------------------------------
seed = DIFFUSION_SEED

print("\n================ SINGLE RUN ================")
print(f"TabDDPM device: {EXPERIMENT_DEVICE}")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if EXPERIMENT_DEVICE.startswith("cuda"):
    gpu_id = int(EXPERIMENT_DEVICE.split(":")[1]) if ":" in EXPERIMENT_DEVICE else 0
    torch.cuda.set_device(gpu_id)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.empty_cache()

train_diffusion_metadata = SingleTableMetadata()
train_diffusion_metadata.detect_from_dataframe(train_real)

if "TabDDPM" in GENERATORS_TO_EVAL:
    import traceback
    try:
        print("Training TabDDPM...")
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=_categorical_columns,
            n_samples=SYNTHETIC_N,
            seed=seed,
            device=EXPERIMENT_DEVICE,
            fast_mode=FAST_MODE,
        )
        synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_diffusion_metadata,
            )
            scores["TabDDPM"] = quality.get_score()
            print("TabDDPM:", round(scores["TabDDPM"], 4))
        else:
            print("TabDDPM: trained (quality eval skipped)")
    except Exception as e:
        print("TabDDPM Failed:", e)
        traceback.print_exc()
else:
    print("TabDDPM: skipped (not in GENERATORS_TO_EVAL)")

if "ForestDiffusion" in GENERATORS_TO_EVAL:
    import traceback
    try:
        print("Training ForestDiffusion...")
        print(f"ForestDiffusion fast_mode={FAST_MODE} (CPU/XGBoost, n_jobs capped)")
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=_categorical_columns,
            n_samples=SYNTHETIC_N,
            seed=seed,
            fast_mode=FAST_MODE,
        )
        synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()
        print("ForestDiffusion: synthesis complete")
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_diffusion_metadata,
            )
            scores["ForestDiffusion"] = quality.get_score()
            print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))
        else:
            print("ForestDiffusion: trained (quality eval skipped)")
    except Exception as e:
        print("ForestDiffusion Failed (training/sampling):")
        traceback.print_exc()
else:
    print("ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)")

synthetic_outputs = {
    name: synthetic_datasets[name].copy()
    for name in ["TabDDPM", "ForestDiffusion"]
    if name in synthetic_datasets
}
model_order = ["TabDDPM", "ForestDiffusion"]
print(f"model_order: {model_order}")


================ SINGLE RUN ================
TabDDPM device: cuda
Training TabDDPM...
[0]
23
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(23)}
mlp
mlp
Sample timestep    0
Discrete cols: [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Num shape:  (1000, 21)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 839.01it/s]|
Column Shapes Score: 76.56%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:00<00:00, 454.92it/s]|
Column Pair Trends Score: 40.52%

Overall Score (Average): 58.54%

TabDDPM: 0.5854
Training ForestDiffusion...
ForestDiffusion fast_mode=True (CPU/XGBoost, n_jobs capped)
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 22/22 [00:00<00:00, 403.35it/s]|
Column Shapes Score: 77.25%

(2/2) Evaluating Column Pair Trends: |██████████| 231/231 [00:00<00:00, 437.13it/s]|
Column 

In [5]:
# Diffusion training moved to the cell above.
print("TabDDPM and ForestDiffusion are trained in the previous cell.")

TabDDPM and ForestDiffusion are trained in the previous cell.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# 10 seeds for TRTR / TSTR evaluation (same as cancer notebook)
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

# All 10 classifiers. FAST_MODE uses cheaper equivalents for slow models (esp. SVM-RBF).
# class_weight="balanced" helps on this imbalanced dataset (~86% class 0).
if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(
            max_iter=500, solver="liblinear", class_weight="balanced", random_state=42
        ),
        # LinearSVC ~100x faster than RBF SVC; keep name for result tables
        "SVM-RBF": LinearSVC(
            max_iter=500, dual="auto", class_weight="balanced", random_state=42
        ),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(
            random_state=42, max_depth=12, class_weight="balanced"
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1, class_weight="balanced"
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1, class_weight="balanced"
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(
            max_iter=5000, solver="liblinear", class_weight="balanced", random_state=42
        ),
        "SVM-RBF": SVC(
            kernel="rbf", cache_size=1000, tol=1e-3, class_weight="balanced", random_state=42
        ),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
        "RandomForest": RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight="balanced"
        ),
        "ExtraTrees": ExtraTreesClassifier(
            random_state=42, n_jobs=-1, class_weight="balanced"
        ),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")

Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [8]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
# This duplicate cell was removed — SVM-RBF (probability=True) caused multi-hour runs.
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def _normalize_labels(y, reference=None):
    """Align label dtypes so sklearn metrics do not mix strings and numbers."""
    y = pd.Series(y).reset_index(drop=True)
    ref = pd.Series(reference).reset_index(drop=True) if reference is not None else y
    ref_numeric = pd.to_numeric(ref, errors="coerce")
    if ref_numeric.notna().all():
        return pd.to_numeric(y, errors="coerce").round().astype(int)
    return y.astype(str)


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
    use_holdout=False,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    y_test_reference = _normalize_labels(test_df[label_col])

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = _normalize_labels(train_df[label_col], reference=y_test_reference)
            X_test = test_df.drop(columns=[label_col])
            y_test = y_test_reference.copy()

            if use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_train_full,
                    y_train_full,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_train_full),
                )

                _, X_test, _, y_test = train_test_split(
                    X_test,
                    y_test,
                    test_size=test_size,
                    random_state=seed,
                    stratify=_safe_stratify(y_test),
                )

            scaler = StandardScaler().fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)
            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train_s, y_train)
            y_pred = _normalize_labels(clf.predict(X_test_s), reference=y_test_reference)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, average="weighted", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, average="weighted", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": _std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": _std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": _std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": _std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {_std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {_std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {_std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {_std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)

In [13]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col
seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real) — 80% train / 20% holdout")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Diffusion generators: {len(model_order)}"
)
print(f"TRTR training set: {train_real.shape} | Holdout test set: {test_real.shape}")
print(f"Classifier models: {list(models.keys())}")
print(f"Diffusion generators: {model_order}")

trtr_results = evaluate_models(
    train_df=train_real,
    test_df=test_real,
    label_col=label_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds,
    use_holdout=True,
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_real,
        label_col=label_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds,
        use_holdout=True,
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout
Classifiers: 10 | Seeds: 10 | Diffusion generators: 2
TRTR training set: (800, 22) | Holdout test set: (200, 22)
Classifier models: ['LogReg', 'SVM-RBF', 'KNN', 'NaiveBayes', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoost', 'AdaBoost', 'MLP']
Diffusion generators: ['TabDDPM', 'ForestDiffusion']
  LogReg...


  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
8,AdaBoost,0.8965 ± 0.0111,0.8801 ± 0.0156,0.8850 ± 0.0164,0.8965 ± 0.0111
0,LogReg,0.8905 ± 0.0050,0.8614 ± 0.0084,0.8876 ± 0.0093,0.8905 ± 0.0050
1,SVM-RBF,0.8815 ± 0.0041,0.8395 ± 0.0072,0.8927 ± 0.0117,0.8815 ± 0.0041
5,RandomForest,0.8800 ± 0.0082,0.8460 ± 0.0129,0.8650 ± 0.0216,0.8800 ± 0.0082
7,GradientBoost,0.8775 ± 0.0082,0.8454 ± 0.0097,0.8608 ± 0.0222,0.8775 ± 0.0082
6,ExtraTrees,0.8725 ± 0.0072,0.8355 ± 0.0115,0.8470 ± 0.0182,0.8725 ± 0.0072
9,MLP,0.8665 ± 0.0088,0.8493 ± 0.0109,0.8432 ± 0.0134,0.8665 ± 0.0088
2,KNN,0.8485 ± 0.0085,0.8121 ± 0.0097,0.7948 ± 0.0156,0.8485 ± 0.0085
4,DecisionTree,0.7940 ± 0.0279,0.7992 ± 0.0187,0.8069 ± 0.0147,0.7940 ± 0.0279
3,NaiveBayes,0.7580 ± 0.1787,0.7696 ± 0.1866,0.8530 ± 0.0119,0.7580 ± 0.1787


TabDDPM - TSTR
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
1,SVM-RBF,0.8650 ± 0.0000,0.8024 ± 0.0000,0.7482 ± 0.0000,0.8650 ± 0.0000
8,AdaBoost,0.8650 ± 0.0000,0.8024 ± 0.0000,0.7482 ± 0.0000,0.8650 ± 0.0000
5,RandomForest,0.8635 ± 0.0034,0.8016 ± 0.0017,0.7480 ± 0.0004,0.8635 ± 0.0034
0,LogReg,0.8625 ± 0.0035,0.8011 ± 0.0018,0.7479 ± 0.0004,0.8625 ± 0.0035
6,ExtraTrees,0.8535 ± 0.0116,0.7966 ± 0.0058,0.7469 ± 0.0014,0.8535 ± 0.0116
2,KNN,0.8455 ± 0.0104,0.7950 ± 0.0082,0.7570 ± 0.0194,0.8455 ± 0.0104
7,GradientBoost,0.8445 ± 0.0180,0.7961 ± 0.0088,0.7585 ± 0.0183,0.8445 ± 0.0180
9,MLP,0.8325 ± 0.0132,0.7879 ± 0.0077,0.7496 ± 0.0113,0.8325 ± 0.0132
4,DecisionTree,0.7595 ± 0.0497,0.7579 ± 0.0288,0.7581 ± 0.0142,0.7595 ± 0.0497
3,NaiveBayes,0.7575 ± 0.0136,0.7584 ± 0.0059,0.7598 ± 0.0056,0.7575 ± 0.0136


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.8920 ± 0.0075,0.8748 ± 0.0075,0.8793 ± 0.0119,0.8920 ± 0.0075
1,SVM-RBF,0.8900 ± 0.0062,0.8623 ± 0.0107,0.8831 ± 0.0100,0.8900 ± 0.0062
8,AdaBoost,0.8845 ± 0.0086,0.8741 ± 0.0085,0.8718 ± 0.0100,0.8845 ± 0.0086
6,ExtraTrees,0.8725 ± 0.0089,0.8391 ± 0.0130,0.8447 ± 0.0194,0.8725 ± 0.0089
5,RandomForest,0.8670 ± 0.0082,0.8318 ± 0.0092,0.8362 ± 0.0222,0.8670 ± 0.0082
2,KNN,0.8645 ± 0.0060,0.8398 ± 0.0087,0.8344 ± 0.0113,0.8645 ± 0.0060
9,MLP,0.8640 ± 0.0137,0.8513 ± 0.0147,0.8453 ± 0.0170,0.8640 ± 0.0137
7,GradientBoost,0.8560 ± 0.0117,0.8357 ± 0.0130,0.8279 ± 0.0162,0.8560 ± 0.0117
4,DecisionTree,0.7695 ± 0.0261,0.7869 ± 0.0220,0.8100 ± 0.0192,0.7695 ± 0.0261
3,NaiveBayes,0.5515 ± 0.2604,0.5708 ± 0.2617,0.8642 ± 0.0163,0.5515 ± 0.2604


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,ForestDiffusion,AdaBoost,1.200000e-02,0.006036,0.013175,1.200000e-02,0.8965 ± 0.0111,0.8845 ± 0.0086
1,ForestDiffusion,LogReg,-1.500000e-03,-0.013488,0.008291,-1.500000e-03,0.8905 ± 0.0050,0.8920 ± 0.0075
2,ForestDiffusion,SVM-RBF,-8.500000e-03,-0.022754,0.009627,-8.500000e-03,0.8815 ± 0.0041,0.8900 ± 0.0062
3,ForestDiffusion,RandomForest,1.300000e-02,0.014277,0.028818,1.300000e-02,0.8800 ± 0.0082,0.8670 ± 0.0082
4,ForestDiffusion,GradientBoost,2.150000e-02,0.009763,0.032963,2.150000e-02,0.8775 ± 0.0082,0.8560 ± 0.0117
5,ForestDiffusion,ExtraTrees,-2.220446e-16,-0.003560,0.002373,-2.220446e-16,0.8725 ± 0.0072,0.8725 ± 0.0089
6,ForestDiffusion,MLP,2.500000e-03,-0.002016,-0.002097,2.500000e-03,0.8665 ± 0.0088,0.8640 ± 0.0137
7,ForestDiffusion,KNN,-1.600000e-02,-0.027691,-0.039624,-1.600000e-02,0.8485 ± 0.0085,0.8645 ± 0.0060
8,ForestDiffusion,DecisionTree,2.450000e-02,0.012244,-0.003106,2.450000e-02,0.7940 ± 0.0279,0.7695 ± 0.0261
9,ForestDiffusion,NaiveBayes,2.065000e-01,0.198780,-0.011190,2.065000e-01,0.7580 ± 0.1787,0.5515 ± 0.2604


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
1,TabDDPM,0.02165,0.043847,0.101392,0.02165
0,ForestDiffusion,0.02540,0.017159,0.003923,0.02540


In [14]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")


Results saved to: TRTR_TSTR_results.xlsx
